# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://github.com/mlcommons/croissant) library. All dataset entities (record sets, fields, columns) are referenced **by their `@id` fields** for reproducibility.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://mlcommons.org/croissant/) available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`. This metadata describes record sets, fields, columns, and other dataset attributes.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset summary description
print(f"Dataset: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}")

## 2. Data Overview
We explore available record sets (`@id`s), and the fields within each, by inspecting the dataset metadata. Record sets define logical groupings/tables in the dataset, with each field representing a column.

We will print all available record sets by `@id`, their names, and their field `@id`s and names.

In [ ]:
# List all available record sets and their fields, using @id when printing
def print_record_sets(ds):
    print('Record sets in the dataset:')
    for rs in ds.metadata.record_sets:
        print(f"- [@id] {rs.id} | [name] {rs.name}")
        print('  Fields:')
        for field in rs.fields:
            print(f"    - [@id] {field.id} | [name] {field.name} | [dataType] {field.data_type}")
        print()

print_record_sets(dataset)

## 3. Data Extraction
Load data from *each* available record set into a pandas DataFrame, referring to each by its `@id`. You can look up each record set/field/column's `@id` from the output above. These DataFrames will be used for further analysis.

> Note: Some record sets may be small metadata tables, others may contain the main tabular data.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print('All record set @ids:', record_set_ids)

# Load all into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set [@id] {rs_id}. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"[WARN] Failed loading record set {rs_id}: {e}")
# For demonstration, select the largest (main) record set. Here we pick the first, but you can adjust as appropriate.main_record_set_id = record_set_ids[0]
print(f"\nUsing [@id] {main_record_set_id} as the main table for EDA.")
print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (by `@id`) from the chosen record set and demonstrate filtering, normalization, and grouping/categorization, as typical first EDA steps.

Replace the IDs below with those for the relevant numeric and grouping fields in your main record set. Use the printed lists above for the correct `@id` values.

In [ ]:
# --- Replace these @id values with the real ones from your dataset schema ---

# Example: For demonstration, we'll attempt to auto-detect a numeric field if possible, else customize per your schema.
df = dataframes[main_record_set_id]

# Try to infer a numeric field by dtype
numeric_fields = df.select_dtypes(include='number').columns.to_list()

if numeric_fields:
    numeric_field_id = numeric_fields[0]   # Use the first numeric field's column/@id
    print(f"Using numeric field (column/@id): {numeric_field_id}")
else:
    print("No numeric fields detected. Please set a numeric field by its @id in your schema.")
    # numeric_field_id = '<numeric_field_id>'

# If needed, set a threshold for filtering
threshold = df[numeric_field_id].mean() if numeric_fields else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}.")

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if available
categorical_fields = df.select_dtypes(include='object').columns.to_list()
if categorical_fields:
    group_field_id = categorical_fields[0]
    print(f"Grouping by {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())
else:
    print("No categorical grouping field found.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field and its normalized values, and demonstrate a simple group-wise bar plot. (Requires `matplotlib` and `seaborn`.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id}')

    plt.subplot(1,2,2)
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True)
    plt.title(f'Normalized {numeric_field_id}')
    plt.tight_layout()
    plt.show()

    if categorical_fields:
        # Barplot of mean values by group
        group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion
We have explored the FAIR² dataset describing protein abundance and characterization in mast cell extracellular vesicles via the `mlcroissant` library. You can further adapt these analysis steps to your research questions by specifying the field `@id`s you need and extending the exploratory analysis as necessary. If your main table contains multiple numeric fields, try plotting or analyzing them by their `@id`s as explored above.


Feel free to extend this notebook with your own domain-specific analyses and deeper dives into the dataset. See the [mlcroissant documentation](https://github.com/mlcommons/croissant) for advanced usage and further automation with schema-based machine learning data packages.